# GEE Analysis — Engagement-Disparity Conformity Effect Across Models

Companion notebook to `docs/THESIS.md` (Chapter 8, item 3) and `docs/STATS_ANALYSIS_NOTES.md`.

**Purpose:** replace the descriptive percentages/heatmaps and per-model sign tests already in the thesis with a single formal statistical model — a binomial GEE (Generalized Estimating Equations) clustered by image — that can test the thesis's central hypotheses directly, with proper handling of the fact that the same 100 images are reused across every grid cell, signal type, and model (see `docs/THESIS.md` Appendix C.1 for a plain-language explanation of why that reuse is a problem, and `docs/STATS_ANALYSIS_NOTES.md` for the method comparison that led to choosing GEE as the starting point).

**Why GEE specifically (not GLMM):** GEE was chosen to start because it converges even when some model/condition combinations produce a constant 0% or 100% answer (several models do, under some phrasings — Section 6.3), which breaks standard maximum-likelihood GLMM fitting via complete separation. GLMM remains a possible follow-up once/if the degenerate cells are handled separately (e.g. dropped, or fit with Firth-penalized regression) — see the pros/cons table in `docs/STATS_ANALYSIS_NOTES.md`.

**Where this runs:** the local dev sandbox used for the rest of this project's Claude Code session has no `pip`/`numpy`/`pandas`/`statsmodels` available and no way to install them. This notebook is meant to be run on your GPU server (or any machine where you can `pip install`), not in that sandbox.

## 0. Setup

Run the install cell once per environment. If `numpy`/`pandas`/`scipy`/`statsmodels` are already available on your server, skip it.

In [1]:
# One-time setup — skip if already installed
!pip install --quiet numpy pandas scipy statsmodels


In [2]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


## 1. Config — models and paths

Adjust `REPO_ROOT` if this notebook is moved or run from a different checkout.

In [3]:
REPO_ROOT = Path("..").resolve()  
assert (REPO_ROOT / "experiments" / "e1").exists(), f"Unexpected REPO_ROOT: {REPO_ROOT}"

MODELS = [
    "gemma4-12b",
    "gemma4-e4b",
    "qwen3-vl-4b",
    "qwen3-vl-8b",
    "pixtral-12b",
    "mistral-small-3.1-24b",
]

# Signal-type conditions that vary engagement scale over a 7x7 grid (correct_scale x incorrect_scale).
# 'baseline' is handled separately below — it shows no engagement numbers at all, so there is no
# disparity to speak of; treating it as "disparity = 0" would silently conflate "no numbers shown"
# with "equal numbers shown", which are different conditions. Revisit this if you disagree.
GRID_CONDITIONS = ["metrics", "likes_only", "likes_only_noise"]

SCALES = [0, 10, 100, 1000, 10000, 100000, 1000000]


## 2. Load the disparity-grid data (metrics / likes_only / likes_only_noise)

Builds one long-format row per trial: which model, which image, the engagement disparity (continuous), which signal type, and whether the model chose the correct post.

**Disparity definition** (matches `docs/STATS_ANALYSIS_NOTES.md`): `disparity = log10(incorrect_scale + 1) - log10(correct_scale + 1)`. Positive disparity means the *incorrect* post has the engagement advantage — this is the direction where conformity bias, if present, should show up as a *lower* probability of choosing correct. Zero means equal engagement (the diagonal of the grid).

In [4]:
def load_grid_condition(model_dir: str, condition: str) -> pd.DataFrame:
    path = REPO_ROOT / "experiments" / "e1" / model_dir / "outputs" / f"e1_results_{condition}_paired.json"
    records = json.loads(path.read_text())
    rows = []
    for r in records:
        cs, ics = r["correct_scale"], r["incorrect_scale"]
        disparity = math.log10(ics + 1) - math.log10(cs + 1)
        rows.append({
            "model": model_dir,
            "image_num": r["num"],
            "signal_type": condition,
            "correct_scale": cs,
            "incorrect_scale": ics,
            "disparity": disparity,
            "chose_correct": 1 if r["liked_variant"] == "correct" else 0,
        })
    return pd.DataFrame(rows)


grid_frames = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        grid_frames.append(load_grid_condition(model_dir, condition))

df = pd.concat(grid_frames, ignore_index=True)
# Cluster unit: same image, reused across all 49 cells x 3 signal types, WITHIN a given model.
# This is the more conservative reading of the pseudo-replication concern in STATS_ANALYSIS_NOTES.md —
# it protects against "this model has an idiosyncratic quirk on this image" being counted many times.
# An alternative (clustering by image_num alone, shared across models) is tried as a robustness check
# in section 5 below — the two should be compared before trusting either one exclusively.
df["cluster"] = df["model"] + "__" + df["image_num"]

print(df.shape)
df.head()


(88200, 8)


,model,image_num,signal_type,correct_scale,incorrect_scale,disparity,chose_correct,cluster
0,gemma4-12b,001,metrics,0,0,0.0,1,gemma4-12b__001
1,gemma4-12b,003,metrics,0,0,0.0,0,gemma4-12b__003
2,gemma4-12b,004,metrics,0,0,0.0,1,gemma4-12b__004
3,gemma4-12b,005,metrics,0,0,0.0,1,gemma4-12b__005
4,gemma4-12b,006,metrics,0,0,0.0,1,gemma4-12b__006


## 3. Sanity check against numbers already in the thesis

Before trusting any model output, reproduce a few descriptive numbers already reported in `docs/THESIS.md` §6.1 (the six-model diagonal/above-diagonal/collapse table) directly from this dataframe. If these don't match, something is wrong with the loading code above — stop and debug before fitting anything.

In [5]:
def diagonal_above_collapse(model_dir: str, condition: str = "metrics"):
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)]
    diag = sub[sub["disparity"] == 0]["chose_correct"].mean() * 100
    above = sub[sub["disparity"] > 0]["chose_correct"].mean() * 100
    return diag, above, diag - above


print(f"{'model':24s} {'diagonal':>10s} {'above-diag':>12s} {'collapse':>10s}")
for m in MODELS:
    d, a, c = diagonal_above_collapse(m)
    print(f"{m:24s} {d:9.1f}% {a:11.1f}% {c:9.1f} pt")

# Expected from THESIS.md Section 6.1 (already verified against raw JSON on 2026-07-15):
#   Gemma-12B    82.7% / 84.7% / -2.0pt (no collapse)
#   Gemma-E4B    52.3% / 1.0%  / 51.2pt
#   Qwen3-VL-4B  45.0% / 9.6%  / 35.4pt
#   Qwen3-VL-8B  55.0% / 1.4%  / 53.6pt
#   Pixtral-12B  51.7% / 22.6% / 29.1pt
#   Mistral-24B  51.7% / 7.5%  / 44.2pt


model                      diagonal   above-diag   collapse
gemma4-12b                    82.7%        84.7%      -2.0 pt
gemma4-e4b                    52.3%         1.0%      51.2 pt
qwen3-vl-4b                   45.0%         9.6%      35.4 pt
qwen3-vl-8b                   55.0%         1.4%      53.6 pt
pixtral-12b                   51.7%        22.6%      29.1 pt
mistral-small-3.1-24b         51.7%         7.5%      44.2 pt


## 4. Primary GEE model

Formula: `chose_correct ~ C(model) * disparity + C(signal_type) * disparity`

- `C(model) * disparity` tests the Section 6.1 scale/family-dependence question directly: does the slope of the disparity effect differ significantly by model, rather than relying on comparing six heatmaps by eye?
- `C(signal_type) * disparity` tests the Section 6.2 metrics-vs-likes-only asymmetry the same way.
- `groups=df["cluster"]` (model+image) and `cov_struct=Independence()` implement the clustering fix described in Appendix C.1/C.2 of the thesis — treats all trials from the same model on the same image as correlated, and computes robust ("sandwich") standard errors accordingly, instead of pretending every row is independent.

This will likely take a little while to fit given ~88,000 rows (6 models x 3 conditions x 4,900 trials each).

In [6]:
gee_model = smf.gee(
    "chose_correct ~ C(model, Treatment(reference='gemma4-12b')) * disparity + C(signal_type, Treatment(reference='metrics')) * disparity",
    groups="cluster",
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence(),
)
gee_result = gee_model.fit()
print(gee_result.summary())


                               GEE Regression Results                              
Dep. Variable:               chose_correct   No. Observations:                88200
Model:                                 GEE   No. clusters:                      600
Method:                        Generalized   Min. cluster size:                 147
                      Estimating Equations   Max. cluster size:                 147
Family:                           Binomial   Mean cluster size:               147.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Wed, 15 Jul 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         22:25:06
                                                                                     coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

### 4a. Clean coefficient table (odds ratios + 95% CI)

Raw logistic-regression coefficients are in log-odds units, which are not intuitive. This converts each coefficient to an odds ratio (`exp(coef)`) with a 95% confidence interval — e.g. an odds ratio of 0.5 for `disparity` means each one-unit increase in disparity (one order of magnitude of engagement favoring the incorrect post) roughly halves the odds of choosing the correct post, holding everything else constant.

In [7]:
def coef_table(result):
    conf = result.conf_int()
    out = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "p_value": result.pvalues,
        "OR": np.exp(result.params),
        "OR_2.5%": np.exp(conf[0]),
        "OR_97.5%": np.exp(conf[1]),
    })
    return out.round(4)


coef_table(gee_result)


,coef,std_err,p_value,OR,OR_2.5%,OR_97.5%
Intercept,1.9396,0.1111,0.0000,6.9559,5.5951,8.6476
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]",-1.9387,0.1158,0.0000,0.1439,0.1147,0.1806
"C(model, Treatment(reference='gemma4-12b'))[T.mistral-small-3.1-24b]",-1.6848,0.1163,0.0000,0.1855,0.1477,0.2330
"C(model, Treatment(reference='gemma4-12b'))[T.pixtral-12b]",-1.8279,0.1176,0.0000,0.1608,0.1277,0.2024
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]",-2.0564,0.1186,0.0000,0.1279,0.1014,0.1614
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]",-1.9464,0.1185,0.0000,0.1428,0.1132,0.1801
"C(signal_type, Treatment(reference='metrics'))[T.likes_only]",-0.0334,0.0238,0.1609,0.9671,0.9230,1.0134
"C(signal_type, Treatment(reference='metrics'))[T.likes_only_noise]",-0.0666,0.0194,0.0006,0.9356,0.9007,0.9718
disparity,-0.3715,0.0297,0.0000,0.6897,0.6506,0.7311
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]:disparity",-0.7420,0.0357,0.0000,0.4762,0.4440,0.5107


## 5. Robustness check — alternate clustering (image only, shared across models)

`docs/STATS_ANALYSIS_NOTES.md` motivates clustering by image because the same 100 images repeat across every cell/condition. Section 4 above clusters by *(model, image)*, on the reasoning that a given model's own idiosyncratic reaction to a given image is the main repeated-measures concern. An arguably more conservative alternative is to cluster by *image alone*, treating any image-level quirk as potentially shared across models too (e.g. a chart that's just visually ambiguous for every model, not only one).

Run both and compare — if the standard errors and significance conclusions are similar either way, the choice doesn't matter much for the substantive conclusions. If they diverge meaningfully, that's worth flagging explicitly rather than picking whichever one supports the preferred story.

In [8]:
gee_model_altcluster = smf.gee(
    "chose_correct ~ C(model, Treatment(reference='gemma4-12b')) * disparity + C(signal_type, Treatment(reference='metrics')) * disparity",
    groups="image_num",  # clustering by image only, pooled across all 6 models
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence(),
)
gee_result_altcluster = gee_model_altcluster.fit()

comparison = pd.DataFrame({
    "coef": gee_result.params,
    "SE (model+image cluster)": gee_result.bse,
    "SE (image-only cluster)": gee_result_altcluster.bse,
})
comparison["SE ratio"] = comparison["SE (image-only cluster)"] / comparison["SE (model+image cluster)"]
comparison.round(4)


,coef,SE (model+image cluster),SE (image-only cluster),SE ratio
Intercept,1.9396,0.1111,0.1091,0.9820
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]",-1.9387,0.1158,0.1115,0.9629
"C(model, Treatment(reference='gemma4-12b'))[T.mistral-small-3.1-24b]",-1.6848,0.1163,0.1194,1.0266
"C(model, Treatment(reference='gemma4-12b'))[T.pixtral-12b]",-1.8279,0.1176,0.1270,1.0803
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]",-2.0564,0.1186,0.1104,0.9311
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]",-1.9464,0.1185,0.1059,0.8941
"C(signal_type, Treatment(reference='metrics'))[T.likes_only]",-0.0334,0.0238,0.0232,0.9748
"C(signal_type, Treatment(reference='metrics'))[T.likes_only_noise]",-0.0666,0.0194,0.0187,0.9622
disparity,-0.3715,0.0297,0.0280,0.9416
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]:disparity",-0.7420,0.0357,0.0346,0.9691


## 6. Formal tests of the thesis's two central hypotheses

Rather than reading individual interaction-term p-values above one at a time, test each hypothesis as a *joint* Wald test across all of that hypothesis's interaction terms together — this is the direct, formal version of what Section 6.1's heatmap-by-heatmap comparison and Section 6.2's metrics-vs-likes-only description currently do by eye/prose.

In [9]:
# Joint test: does the disparity slope differ significantly across models at all?
# (i.e. is there any real model x disparity interaction, not just "models look different on a heatmap")
model_disparity_terms = [name for name in gee_result.params.index if "C(model" in name and "disparity" in name and ":" in name]
print("Terms included in the model x disparity joint test:")
for t in model_disparity_terms:
    print(" -", t)

wald_model_disparity = gee_result.wald_test(model_disparity_terms, scalar=True)
print("\nJoint Wald test (model x disparity):", wald_model_disparity)


Terms included in the model x disparity joint test:
 - C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.mistral-small-3.1-24b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.pixtral-12b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]:disparity
 - C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]:disparity

Joint Wald test (model x disparity): <Wald test (chi2): statistic=3267.5338186386944, p-value=0.0, df_denom=5>


In [10]:
# Joint test: does the disparity slope differ significantly by signal type?
# (the Section 6.2 metrics-vs-likes-only-vs-noise asymmetry, tested formally instead of by eye)
signal_disparity_terms = [name for name in gee_result.params.index if "C(signal_type" in name and ":disparity" in name]
print("Terms included in the signal_type x disparity joint test:")
for t in signal_disparity_terms:
    print(" -", t)

wald_signal_disparity = gee_result.wald_test(signal_disparity_terms, scalar=True)
print("\nJoint Wald test (signal_type x disparity):", wald_signal_disparity)


Terms included in the signal_type x disparity joint test:
 - C(signal_type, Treatment(reference='metrics'))[T.likes_only]:disparity
 - C(signal_type, Treatment(reference='metrics'))[T.likes_only_noise]:disparity

Joint Wald test (signal_type x disparity): <Wald test (chi2): statistic=550.825337086571, p-value=2.4535663421926882e-120, df_denom=2>


## 7. Baseline condition (no engagement shown at all) — separate, simpler model

The `baseline` condition shows no engagement numbers at all, so it has no `disparity` value — it doesn't belong in the grid model above. This is the same comparison already covered descriptively in `docs/THESIS.md` (each model's unscaled A/B baseline, e.g. Gemma-12B 83%/17%), reproduced here as a single clustered logistic model across all six models for a formal cross-model comparison with a p-value, rather than six numbers compared by eye. There is only one trial per image per model in this condition (no repeated cells), so clustering only matters if you consider the same image across the 6 models as correlated — included for consistency, but the correction will be minor here.

In [11]:
def load_baseline(model_dir: str) -> pd.DataFrame:
    path = REPO_ROOT / "experiments" / "e1" / model_dir / "outputs" / "e1_results_baseline_paired.json"
    records = json.loads(path.read_text())
    rows = [{
        "model": model_dir,
        "image_num": r["num"],
        "chose_correct": 1 if r["liked_variant"] == "correct" else 0,
    } for r in records]
    return pd.DataFrame(rows)


baseline_df = pd.concat([load_baseline(m) for m in MODELS], ignore_index=True)

baseline_gee = smf.gee(
    "chose_correct ~ C(model, Treatment(reference='gemma4-12b'))",
    groups="image_num",
    data=baseline_df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Independence(),
)
baseline_result = baseline_gee.fit()
coef_table(baseline_result)


,coef,std_err,p_value,OR,OR_2.5%,OR_97.5%
Intercept,1.5856,0.2662,0.0000,4.8824,2.8975,8.2269
"C(model, Treatment(reference='gemma4-12b'))[T.gemma4-e4b]",-1.1383,0.3748,0.0024,0.3204,0.1537,0.6679
"C(model, Treatment(reference='gemma4-12b'))[T.mistral-small-3.1-24b]",-1.3445,0.3781,0.0004,0.2607,0.1242,0.5470
"C(model, Treatment(reference='gemma4-12b'))[T.pixtral-12b]",-1.3445,0.3781,0.0004,0.2607,0.1242,0.5470
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-4b]",-2.2047,0.3204,0.0000,0.1103,0.0589,0.2067
"C(model, Treatment(reference='gemma4-12b'))[T.qwen3-vl-8b]",-1.4655,0.2872,0.0000,0.2310,0.1315,0.4056


## 8. Export results

Saves the primary model's coefficient table and both Wald test results to `statistical_analysis/outputs/` for pasting into `docs/THESIS.md` / `docs/STATS_ANALYSIS_NOTES.md` once reviewed. Nothing here is written back into the thesis automatically — treat this as a draft result to check against the existing descriptive findings (Section 3 above) before it replaces or supplements anything in Chapter 6.

In [12]:
out_dir = REPO_ROOT / "statistical_analysis" / "outputs"
out_dir.mkdir(exist_ok=True)

coef_table(gee_result).to_csv(out_dir / "gee_primary_model_coefficients.csv")
comparison.to_csv(out_dir / "gee_clustering_robustness_check.csv")
coef_table(baseline_result).to_csv(out_dir / "gee_baseline_model_coefficients.csv")

with open(out_dir / "wald_tests.txt", "w") as f:
    f.write("Joint Wald test (model x disparity):\n")
    f.write(str(wald_model_disparity) + "\n\n")
    f.write("Joint Wald test (signal_type x disparity):\n")
    f.write(str(wald_signal_disparity) + "\n")

print(f"Saved outputs to {out_dir}")


Saved outputs to /home/jovyan/conformity-llms-facebook-posts/statistical_analysis/outputs


## 9. Formal test of the recurring high-magnitude discrimination-collapse pattern (Section 6.4)

`docs/THESIS.md` Section 6.4 documents a pattern, found by inspecting grid slices by eye, in which five of the eighteen model x signal-type combinations show accuracy converging toward chance once *both* posts' engagement counts get large (roughly 10,000+), regardless of which post is favored — while the other thirteen combinations stay confidently saturated at every scale. This section tests that pattern formally instead of continuing to eyeball heatmap slices.

**Why `disparity` alone (or `disparity**2`) cannot detect this.** `disparity` is a log-*ratio* (`log10(incorrect+1) - log10(correct+1)`), so it is scale-invariant by construction: the pair "10 vs. 100" and the pair "10,000 vs. 100,000" have the *identical* disparity value, even though the collapse only shows up in the second, much larger pair. A squared-disparity term is still only a function of the ratio, so it cannot pick this up either.

**The fix: add a separate term for the absolute scale of the numbers shown, and test its interaction with disparity.** `avg_scale = (log10(correct_scale + 1) + log10(incorrect_scale + 1)) / 2` captures how large the two numbers are, independent of their ratio (0 at "0 vs. 0", up to 6 at "1,000,000 vs. 1,000,000"). Fitting `chose_correct ~ disparity * avg_scale` **separately for each of the 18 model x signal-type combinations** lets us read off, for each one, whether the `disparity:avg_scale` interaction is significantly negative — i.e. whether the disparity effect genuinely weakens as the numbers get larger. That is the direct, formal test of what Section 6.4 currently only shows via two representative grid slices per flagged combination.

In [ ]:
df["avg_scale"] = (
    np.log10(df["correct_scale"] + 1) + np.log10(df["incorrect_scale"] + 1)
) / 2

# Sanity check: avg_scale should range from 0 (0 vs 0) to 6 (1M vs 1M)
print(df["avg_scale"].min(), df["avg_scale"].max())
df[["model", "signal_type", "correct_scale", "incorrect_scale", "disparity", "avg_scale"]].head()


In [ ]:
scale_interaction_rows = []

for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)].copy()
        try:
            m = smf.gee(
                "chose_correct ~ disparity * avg_scale",
                groups="image_num",
                data=sub,
                family=sm.families.Binomial(),
                cov_struct=sm.cov_struct.Independence(),
            )
            r = m.fit()
            term = "disparity:avg_scale"
            scale_interaction_rows.append({
                "model": model_dir,
                "signal_type": condition,
                "coef": r.params[term],
                "std_err": r.bse[term],
                "p_value": r.pvalues[term],
                "n": len(sub),
                "converged": True,
            })
        except Exception as e:
            scale_interaction_rows.append({
                "model": model_dir,
                "signal_type": condition,
                "coef": np.nan,
                "std_err": np.nan,
                "p_value": np.nan,
                "n": len(sub),
                "converged": False,
                "error": str(e),
            })
            print(f"FAILED to fit {model_dir} / {condition}: {e}")

scale_interaction = pd.DataFrame(scale_interaction_rows).sort_values("p_value")
scale_interaction


### 9a. Compare against the five combinations flagged descriptively in Section 6.4

Flagged by eye (Section 6.4, 2026-07-16): Qwen3-VL-4B/`metrics`, Gemma-E4B/`likes_only`, Gemma-E4B/`likes_only_noise`, Qwen3-VL-8B/`likes_only`, Mistral Small 3.1 24B/`likes_only`.

A combination "confirms" the descriptive flag here if its `disparity:avg_scale` coefficient is negative **and** significant (p < 0.05) — meaning the disparity effect formally, measurably weakens as the shown numbers get larger. Combinations flagged by eye but *not* confirmed here are worth a second look (the eye-based check might have been swayed by a couple of specific slices rather than the overall pattern); combinations confirmed here but *not* flagged by eye would mean the descriptive pass under-counted the pattern.

In [ ]:
flagged_by_eye = {
    ("qwen3-vl-4b", "metrics"),
    ("gemma4-e4b", "likes_only"),
    ("gemma4-e4b", "likes_only_noise"),
    ("qwen3-vl-8b", "likes_only"),
    ("mistral-small-3.1-24b", "likes_only"),
}

scale_interaction["flagged_by_eye"] = scale_interaction.apply(
    lambda row: (row["model"], row["signal_type"]) in flagged_by_eye, axis=1
)
scale_interaction["confirmed_by_gee"] = (scale_interaction["coef"] < 0) & (scale_interaction["p_value"] < 0.05)

comparison_table = scale_interaction[[
    "model", "signal_type", "coef", "p_value", "flagged_by_eye", "confirmed_by_gee"
]].sort_values(["flagged_by_eye", "p_value"], ascending=[False, True])
comparison_table


In [ ]:
out_path = REPO_ROOT / "statistical_analysis" / "outputs" / "scale_interaction_test.csv"
comparison_table.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


## 10. Opposite-corner-sum test (supervisor suggestion, 2026-07-16)

**The idea.** Take a grid cell `g(X, Y) = P(chose correct | correct_scale=X, incorrect_scale=Y)` and its mirror cell `g(Y, X)` (the same pair of engagement numbers, but swapped between the correct and incorrect post). If the model's choice were driven *purely* by which post has more engagement — with zero independent weight given to which post is actually correct — then swapping the numbers should just swap the winner, and `g(X,Y) + g(Y,X)` should equal exactly 100%. A sum **meaningfully above 100%** means correctness is pulling extra weight on top of the engagement effect in both cells — direct evidence of a genuine, independent correctness effect, not just engagement-following.

**Method.** For each model x signal-type combination, and for every unique off-diagonal pair of engagement scales `{X, Y}` (21 pairs; the diagonal `X == Y` is excluded since a cell is trivially its own mirror there), compute a paired per-image quantity: for each of the 100 images, `s_img = chose_correct(X,Y) + chose_correct(Y,X)` (a 0, 1, or 2 for that image at that scale pair). Under the "pure engagement, zero correctness effect" null, the population-level expectation of `s_img` is 1 (matching the 100% sum). This is tested the same way Section 6.1 already tests the competence/collapse comparison — an exact sign test: count images where `s_img > 1` ("excess correctness," win) vs. `s_img < 1` ("correctness deficit," loss), excluding ties (`s_img == 1`), and test whether that split could plausibly arise by chance.

In [ ]:
from scipy.stats import binomtest

def opposite_corner_sign_test(model_dir: str, condition: str):
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition)]
    # index by (image_num, correct_scale, incorrect_scale) -> chose_correct
    lookup = {
        (r["image_num"], r["correct_scale"], r["incorrect_scale"]): r["chose_correct"]
        for r in sub.to_dict("records")
    }
    wins = losses = ties = 0
    sums = []
    seen_pairs = set()
    for cs in SCALES:
        for ics in SCALES:
            if cs == ics:
                continue
            pair_key = frozenset({cs, ics})
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
            for image_num in sub["image_num"].unique():
                a = lookup.get((image_num, cs, ics))
                b = lookup.get((image_num, ics, cs))
                if a is None or b is None:
                    continue
                s = a + b
                sums.append(s)
                if s > 1:
                    wins += 1
                elif s < 1:
                    losses += 1
                else:
                    ties += 1
    n_decisive = wins + losses
    if n_decisive == 0:
        return None
    test = binomtest(wins, n_decisive, 0.5)
    return {
        "model": model_dir,
        "signal_type": condition,
        "mean_sum_pct": 100 * sum(sums) / len(sums),
        "n_wins_excess_correctness": wins,
        "n_losses_deficit": losses,
        "n_ties": ties,
        "p_value": test.pvalue,
    }


corner_rows = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        r = opposite_corner_sign_test(model_dir, condition)
        if r is not None:
            corner_rows.append(r)

corner_results = pd.DataFrame(corner_rows).sort_values("p_value")
corner_results


**Reading this table:** `mean_sum_pct` well above 100%, paired with a small `p_value`, is evidence of a genuine correctness effect independent of engagement. A `mean_sum_pct` close to 100% (regardless of p-value) is consistent with the "pure engagement, no independent correctness weight" null — i.e. more evidence *against* the model doing any real fact-checking on top of following the crowd.

In [ ]:
out_path = REPO_ROOT / "statistical_analysis" / "outputs" / "opposite_corner_test.csv"
corner_results.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


## 11. Diagonal > 50% test (supervisor suggestion, 2026-07-16)

Extends the per-image diagonal ("competence") scores already used in `THESIS.md` Section 6.1 with an explicit one-sample significance test against chance (50%), rather than only the paired diagonal-vs-above-diagonal comparison already run there. For each model x signal-type combination: average each image's accuracy across its 7 equal-engagement (diagonal) cells, then run an exact sign test on whether each image's diagonal score is above or below 50% (ties at exactly 50% excluded).

In [ ]:
def diagonal_above_chance_test(model_dir: str, condition: str):
    sub = df[(df["model"] == model_dir) & (df["signal_type"] == condition) & (df["disparity"] == 0)]
    per_image = sub.groupby("image_num")["chose_correct"].mean()
    wins = int((per_image > 0.5).sum())
    losses = int((per_image < 0.5).sum())
    ties = int((per_image == 0.5).sum())
    n_decisive = wins + losses
    if n_decisive == 0:
        return None
    test = binomtest(wins, n_decisive, 0.5)
    return {
        "model": model_dir,
        "signal_type": condition,
        "mean_diagonal_pct": 100 * per_image.mean(),
        "n_images_above_50": wins,
        "n_images_below_50": losses,
        "n_images_at_50": ties,
        "p_value": test.pvalue,
    }


diagonal_rows = []
for model_dir in MODELS:
    for condition in GRID_CONDITIONS:
        r = diagonal_above_chance_test(model_dir, condition)
        if r is not None:
            diagonal_rows.append(r)

diagonal_results = pd.DataFrame(diagonal_rows).sort_values("p_value")
diagonal_results


In [ ]:
out_path = REPO_ROOT / "statistical_analysis" / "outputs" / "diagonal_above_chance_test.csv"
diagonal_results.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
